# BİM448 Veri Madenciliği ve Bilgi Keşfi - Proje Ödevi
## 1. Yöntem: VADER Duygu Analizi (Kural Tabanlı Yaklaşım)
- **İngilizce:** Doğrudan VADER (`SentimentIntensityAnalyzer`) uygulaması (9.000 yorum)
- **Türkçe:** Google Translate ile İngilizceye çeviri + VADER uygulaması ve çevirisiz ham VADER ile karşılaştırma (9.000 yorum)

In [1]:
!pip install -q vaderSentiment deep-translator scikit-learn pandas
import os
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

LABELS = ["Positive", "Neutral", "Negative"]
def get_sentiment_label(compound: float) -> str:
    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

In [2]:
# 1. İNGİLİZCE VADER ANALİZİ
df_en = pd.read_csv("../Veri_Setleri/english_restaurant_reviews_9000.csv")
analyzer = SentimentIntensityAnalyzer()
df_en["vader_compound"] = [analyzer.polarity_scores(str(t))["compound"] for t in df_en["text"]]
df_en["vader_pred"] = [get_sentiment_label(c) for c in df_en["vader_compound"]]
print(classification_report(df_en["true_label"], df_en["vader_pred"], labels=LABELS, digits=4))
print(pd.DataFrame(confusion_matrix(df_en["true_label"], df_en["vader_pred"], labels=LABELS),
                   index=[f"Gerçek {l}" for l in LABELS], columns=[f"Tahmin {l}" for l in LABELS]))

In [3]:
# 2. TÜRKÇE VADER ANALİZİ (Çevirili vs Çevirisiz)
df_tr = pd.read_csv("vader_turkish_results.csv")
print("=== ÇEVİRİLİ TÜRKÇE VADER SONUÇLARI ===")
print(classification_report(df_tr["true_label"], df_tr["vader_pred"], labels=LABELS, digits=4))
print(pd.DataFrame(confusion_matrix(df_tr["true_label"], df_tr["vader_pred"], labels=LABELS),
                   index=[f"Gerçek {l}" for l in LABELS], columns=[f"Tahmin {l}" for l in LABELS]))

1) İngilizce veri seti (9.000 yorum) üzerinde VADER çalıştırılıyor...
   Tamamlandı (1.26 sn) -> vader_english_results.csv kaydedildi.

 İNGİLİZCE VERİ SETİ - VADER SONUÇLARI (9.000 Yorum)
Genel Doğruluk (Accuracy)   : %51.53 (4,638 / 9,000 doğru)
Toplam Hatalı Tahmin Sayısı : 4,362 / 9,000

Sınıf Bazlı Hata Analizi (Gerçek Sınıfa Göre):
  - [Positive] Toplam: 3000 | Doğru: 2936 | Hatalı:   64 (% 2.13 hata) -> (Neutral'e kayan: 20, Negative'e kayan: 44)
  - [Neutral ] Toplam: 3000 | Doğru:   79 | Hatalı: 2921 (%97.37 hata) -> (Positive'e kayan: 2447, Negative'e kayan: 474)
  - [Negative] Toplam: 3000 | Doğru: 1623 | Hatalı: 1377 (%45.90 hata) -> (Positive'e kayan: 1247, Neutral'e kayan: 130)

Karmaşıklık Matrisi (Confusion Matrix):
                 Tahmin Positive  Tahmin Neutral  Tahmin Negative
Gerçek Positive             2936              20               44
Gerçek Neutral              2447              79              474
Gerçek Negative             1247             130            